# 第 10 章 オークンの法則とフィリップス曲線: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-programming/10-okun-and-phillips.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
smoothing = 1600   # HP フィルターの平滑化の強さ λ
split_year = 2000  # フィリップス曲線を、この年の前後で分ける
window = 40        # 傾きの推移を見る、窓の長さ（四半期）

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
sys.path[:] = [p for p in sys.path if p != str(_root)]
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and Path(getattr(_lib, "__file__", "") or "").resolve() != (_root / "lib" / "__init__.py").resolve():
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 回帰の傾き

In [ ]:
from lib.chapter09.cycles import load_quarterly
from lib.chapter10.okun_phillips import (
    okun_regression_from_data,
    phillips_by_period,
    phillips_regression_from_data,
)

df = load_quarterly()
okun = okun_regression_from_data(df, smoothing)
print("オークンの法則の傾き:", round(float(okun.coefficients["output_cycle"]), 3))
phillips = phillips_regression_from_data(df, smoothing)
print("フィリップス曲線の傾き（全期間）:",
      round(float(phillips.coefficients["unemployment_gap"]), 3))
parts = phillips_by_period(df, split_year, smoothing)
labels = {"before": f"{split_year} 年より前", "after": f"{split_year} 年以降"}
for name, label in labels.items():
    slope = float(parts[name].coefficients["unemployment_gap"])
    print(f"{label}の傾き: {slope:.3f}")

## 図: 傾きの推移（窓をずらして、回帰する）

In [ ]:
import pandas as pd

from lib.chapter09.cycles import cycle
from lib.chapter10.okun_phillips import rolling_slope
from lib.chapter10.plot import plot_rolling_slope

plot_rolling_slope(
    rolling_slope(
        cycle(pd.Series(df["unemployment"]), smoothing),
        pd.Series(df["inflation"]),
        window=window,
    )
)

## 試してみよう

- `split_year = 1995` にすると、フィリップス曲線の傾きは、1995 年より前が −0.504、
  1995 年以降が −0.241 です。分ける年を変えても、**あとのほうが平ら**になります
- `smoothing = 100` にすると、オークンの法則の傾きは −0.492、
  フィリップス曲線の傾き（全期間）は −0.351 です（HP フィルターの強さで、
  傾きは少し変わります）